In [1]:
import pandas as pd
import json

# Load the JSON file
with open("./bosses/65-88888801/team_members.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert the list to DataFrame
df = pd.DataFrame(data["list"])

# Extract the number before the last comma from 'pids' and assign it to 'pid'
df["pid"] = df["pids"].apply(
    lambda x: x.rsplit(",", 2)[-2] if isinstance(x, str) and "," in x else None
)

# Add 'layer' column = number of commas in 'pids'
df["layer"] = df["pids"].apply(
    lambda x: x.count(",") if isinstance(x, str) else 0
)


In [2]:
# 1) Filter to layer == 3
df_layer3 = df[df['layer'] == 3].copy()
df_layer2 = df[df['layer'] == 2].copy()
# 2) Join filtered df with original on pid and userId
#    We want to see which original rows do NOT have a match in the layer-3 subset.
joined = df_layer3.merge(
    df_layer2[['pid', 'userId']].drop_duplicates(),
    on=['pid', 'userId'],
    how='left',
    indicator=True
)

# 3) Unmatched records: rows in original that have no (pid, userId) in layer==3
unmatched = joined[joined['_merge'] == 'left_only'].drop(columns=['_merge'])

print(unmatched['pid'].unique())

<StringArray>
['81144275', '77193005', '67198819', '97757906', '39252438']
Length: 5, dtype: str


In [22]:
row_counts = (
    pd.Series({
        s: df['pids'].str.contains(s, regex=False, na=False).sum() 
        for s in unmatched['pid'].unique()
    })
    .sort_values(ascending=False)
)

print(row_counts)

70189747    2472
19826891    1398
89777212     725
13857930     683
89272530     512
42553436     510
65706663     336
15873350     268
37780954     149
64469676     129
52932874      96
35399162      82
84118793      70
14988750      57
50730109      40
17956316      13
96220972      13
26206957      10
34144629       9
77834701       6
31133624       6
37556327       4
dtype: int64


In [30]:
mapping_string = ', '.join(
    f"{row['count']}:'{row['pid']}'" 
    for _, row in (
        row_counts
        .reset_index(name='count')
        .rename(columns={'index': 'pid'})
        .drop_duplicates(subset=['count'], keep=False)
        .iterrows()
    )
)
# Result: "pid1:3, pid5:4, pid7:5"
print(mapping_string)

2472:'70189747', 1398:'19826891', 725:'89777212', 683:'13857930', 512:'89272530', 510:'42553436', 336:'65706663', 268:'15873350', 149:'37780954', 129:'64469676', 96:'52932874', 82:'35399162', 70:'84118793', 57:'14988750', 40:'50730109', 10:'26206957', 9:'34144629', 4:'37556327'


In [44]:
# 1) Filter to layer == 3
df_layer3 = df[df['layer'] == 4].copy()
df_layer2 = df[df['layer'] == 3].copy()
# 2) Join filtered df with original on pid and userId
#    We want to see which original rows do NOT have a match in the layer-3 subset.
joined = df_layer3.merge(
    df_layer2[['pid', 'userId']].drop_duplicates(),
    on=['pid', 'userId'],
    how='left',
    indicator=True
)

# 3) Unmatched records: rows in original that have no (pid, userId) in layer==3
unmatched = joined[joined['_merge'] == 'left_only'].drop(columns=['_merge'])

print(unmatched['pid'].unique())

<StringArray>
['76840891', '23716124', '92295564', '68088732', '32290201', '64484086',
 '47766662', '26997858', '34125512', '78968576', '36027415', '17329192',
 '69372126', '49891835', '79884106', '88992492', '16863824', '37887523',
 '36308517', '74805483', '87019332', '86098813', '18391289', '75955528',
 '18631739', '27150347', '56045241', '91936992', '83520417', '19681267',
 '70943631', '83962112', '78293011', '37346251', '21274166', '14071688',
 '29845098', '96690894', '55796345', '80309824', '47557322', '22828467',
 '21720997', '25803316', '69062661', '57771618', '60897180', '41765860',
 '26976920', '59117095', '21326453', '20865453', '98031708']
Length: 53, dtype: str


In [28]:
print(df_layer3)

          id    userId  ...                                 pids layer
171    71029  60692776  ...  88888888,86270534,34636784,32290201     3
523    70676  84414822  ...  88888888,86270534,34636784,32290201     3
1159   70038  33721283  ...  88888888,86270534,34636784,32290201     3
3115   68081  84380358  ...  88888888,86270534,34636784,32290201     3
4604   66592  90468041  ...  88888888,86270534,39472923,70943631     3
...      ...       ...  ...                                  ...   ...
31995   9487  80743997  ...  88888888,86270534,39472923,29845098     3
31996   9486  88026553  ...  88888888,86270534,39472923,18391289     3
31997   9485  70185035  ...  88888888,86270534,16785441,76840891     3
31998   9484  46923144  ...  88888888,86270534,34636784,68088732     3
31999   9483  16939229  ...  88888888,86270534,32033515,80309824     3

[285 rows x 29 columns]


In [3]:
my_list = ['97757906']
row_counts = (
    pd.Series({
        s: df['pids'].str.contains(s, regex=False, na=False).sum() 
        for s in my_list
    })
    .sort_values(ascending=False)
)

print(row_counts)

97757906    625
dtype: int64


In [41]:
# Convert Series to DataFrame, assign column names, then export to list of dicts
list_of_dicts = (
    row_counts
    .reset_index(name='count')
    .rename(columns={'index': 'pid'})
    .drop_duplicates(subset=['count'], keep=False)
    .to_dict(orient='records')
)

print(list_of_dicts)

[{'pid': '76840891', 'count': 21356}, {'pid': '96690894', 'count': 1458}, {'pid': '87019332', 'count': 1222}, {'pid': '23716124', 'count': 1054}, {'pid': '17329192', 'count': 682}, {'pid': '26997858', 'count': 676}, {'pid': '92295564', 'count': 674}, {'pid': '19681267', 'count': 633}, {'pid': '32290201', 'count': 509}, {'pid': '68088732', 'count': 469}, {'pid': '59117095', 'count': 417}, {'pid': '55796345', 'count': 382}, {'pid': '74805483', 'count': 272}, {'pid': '64484086', 'count': 246}, {'pid': '69372126', 'count': 230}, {'pid': '75955528', 'count': 180}, {'pid': '14071688', 'count': 169}, {'pid': '91936992', 'count': 134}, {'pid': '83962112', 'count': 120}, {'pid': '47766662', 'count': 118}, {'pid': '18391289', 'count': 109}, {'pid': '79884106', 'count': 92}, {'pid': '29845098', 'count': 71}, {'pid': '18631739', 'count': 70}, {'pid': '70943631', 'count': 60}, {'pid': '37887523', 'count': 56}, {'pid': '86098813', 'count': 50}, {'pid': '36027415', 'count': 49}, {'pid': '88992492', '

In [46]:
count_to_pid = {
    item['count']: item['pid']
    for item in list_of_dicts
}

In [47]:
print (count_to_pid)

{21356: '76840891', 1458: '96690894', 1222: '87019332', 1054: '23716124', 682: '17329192', 676: '26997858', 674: '92295564', 633: '19681267', 509: '32290201', 469: '68088732', 417: '59117095', 382: '55796345', 272: '74805483', 246: '64484086', 230: '69372126', 180: '75955528', 169: '14071688', 134: '91936992', 120: '83962112', 118: '47766662', 109: '18391289', 92: '79884106', 71: '29845098', 70: '18631739', 60: '70943631', 56: '37887523', 50: '86098813', 49: '36027415', 45: '88992492', 38: '57771618', 33: '26976920', 31: '56045241', 27: '98031708', 14: '80309824', 8: '78293011', 7: '36308517', 6: '83520417', 4: '27150347'}
